<a href="https://colab.research.google.com/github/lorenzobalzani/nlp_assigments/blob/master/nlp_project_emotion_trigger_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-task neural network for emotion classification and trigger extraction in multi-utterances dialogs
## Project report for the Natural Language Processing course

### Authors: Lorenzo Balzani, Thomas Guizzetti, and Alessia Deana.

# Setup

In [19]:
%%capture
%pip install evaluate transformers[torch] huggingface_hub gdown
!sudo apt-get install git-lfs # for hugging-face hub

import gdown
import json
import pandas as pd
from typing import List, Dict, Tuple
from tqdm import tqdm

from transformers import BertModel, BertTokenizer, AutoModel
from huggingface_hub import PyTorchModelHubMixin
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, AutoTokenizer, EarlyStoppingCallback, pipeline, get_linear_schedule_with_warmup
from datasets import Dataset, DatasetDict, load_dataset
import evaluate

from collections import Counter
from itertools import chain

import seaborn as sns
import matplotlib.pyplot as plt

random_state: int = 42

## Optional

In [2]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Token: 
Add token as git credential? (Y/n) n
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


# Dataset

## Download and pre-processing

In [3]:
training_set_filename = f"MELD_train_efr.json"

try:
  print(f"Attempting to download {training_set_filename} from Drive...")
  file_id = '1wVNU2XvvhqjaGXZM-JLJwOt97gt4g9j2'  # Extracted File ID from your provided URL
  url = f'https://drive.google.com/uc?id={file_id}'
  gdown.download(url, training_set_filename, quiet=False)
  with open(training_set_filename, 'r') as file:
        training_set_json = json.load(file)

  print(f"\nSuccessfully downloaded {training_set_filename} from Drive!")
except:
  print(f"Error loading {training_set_filename} set from Drive")

Attempting to download MELD_train_efr.json from Drive...


Downloading...
From: https://drive.google.com/uc?id=1wVNU2XvvhqjaGXZM-JLJwOt97gt4g9j2
To: /content/MELD_train_efr.json
100%|██████████| 5.18M/5.18M [00:00<00:00, 258MB/s]



Successfully downloaded MELD_train_efr.json from Drive!


In [4]:
df = pd.DataFrame(training_set_json)

# Check the number of NaN values within the lists in the 'triggers' column
nan_count_before = df['utterances'].apply(lambda x: pd.Series(x).isna().sum()).sum()

# Display the number of NaN values
print(f"Number of NaN values in the 'triggers' column lists: {nan_count_before}")

# Replace NaN values within each list in the 'triggers' column with 0.0
df['triggers'] = df['triggers'].apply(lambda x: [0.0 if pd.isna(val) else val for val in x])

# Check again the number of NaN values within the lists in the 'triggers' column
nan_count_after = df['triggers'].apply(lambda x: pd.Series(x).isna().sum()).sum()

# Display the number of NaN values after replacing with 0s
print(f"Number of NaN values in the 'triggers' column lists after replacing: {nan_count_after}")

Number of NaN values in the 'triggers' column lists: 0
Number of NaN values in the 'triggers' column lists after replacing: 0


In [5]:
train_ratio: float = 0.8
val_ratio: float = 0.1
test_ratio: float = 0.1
columns_to_keep: List[str] = ["utterances", "emotions", "triggers"]

labels_emotions = [label for label in df.explode("emotions")["emotions"].unique()]
id2label_emotions = {idx:label for idx, label in enumerate(labels_emotions)}
label2id_emotions = {label:idx for idx, label in enumerate(labels_emotions)}

df["emotions"] = df["emotions"].apply(lambda emotions: [label2id_emotions[emotion] for emotion in emotions])

train_data, temp_data = train_test_split(df, test_size=(1-train_ratio), random_state=random_state)
eval_data, test_data = train_test_split(temp_data, test_size=(test_ratio / (val_ratio + test_ratio)), random_state=random_state)
train_data = train_data[columns_to_keep].reset_index(drop=True)
eval_data = eval_data[columns_to_keep].reset_index(drop=True)
test_data = test_data[columns_to_keep].reset_index(drop=True)

try:
  dataset_hf = DatasetDict({
      "train":  Dataset.from_pandas(train_data),
      "test": Dataset.from_pandas(test_data),
      "eval": Dataset.from_pandas(eval_data),
  })
  dataset_hf.push_to_hub("balzanilo/dialogs-mtl-dataset", private=False)
except Exception:
  print("Error: you must be logged on the HuggingFace Hub to upload the dataset.")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/638 [00:00<?, ?B/s]

## PyTorch dataset adaptation

In [6]:
class EmotionTriggerDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_tokens_per_sentence: int, pad_targets: bool = False):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_tokens_per_sentence = max_tokens_per_sentence
        self.__pad_targets = pad_targets

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if type(idx) == list:
          idx = idx[0] # ONLY works with batch_size = 1
        tokenized_dialogue = self.tokenizer(self.dataset.iloc[idx]["utterances"], truncation=True, padding='max_length', max_length=self.max_tokens_per_sentence, return_tensors='pt')

        emotions = self.dataset.iloc[idx]['emotions']
        triggers = self.dataset.iloc[idx]['triggers']

        if self.__pad_targets:
          emotion_labels = torch.nn.functional.pad(torch.tensor(emotions), (0, self.max_tokens_per_sentence - len(emotions)), value=-1)
          trigger_labels = torch.nn.functional.pad(torch.tensor(triggers), (0, self.max_tokens_per_sentence - len(triggers)), value=-1)
        else:
          emotion_labels = emotions
          trigger_labels = triggers

        return {'input_ids': tokenized_dialogue['input_ids'],
                'attention_mask': tokenized_dialogue['attention_mask'],
                'emotions': torch.LongTensor(emotion_labels),
                'triggers': torch.FloatTensor(trigger_labels)}

In [23]:
encoder_model_name = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(encoder_model_name)
max_tokens_per_sentence: int = 25
train_dataset = EmotionTriggerDataset(train_data, roberta_tokenizer, max_tokens_per_sentence=max_tokens_per_sentence)
eval_dataset = EmotionTriggerDataset(eval_data, roberta_tokenizer, max_tokens_per_sentence=max_tokens_per_sentence)
test_dataset = EmotionTriggerDataset(test_data, roberta_tokenizer, max_tokens_per_sentence=max_tokens_per_sentence)
train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
eval_dataloader = DataLoader(eval_dataset, batch_size=1, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [8]:
for batch_data in train_dataloader:
    # Extract information about the batch
    n_sentences = batch_data["input_ids"].shape[0]
    n_tokens = batch_data["input_ids"].shape[-1]
    n_emotions = batch_data["emotions"].shape[0]
    n_triggers = batch_data["triggers"].shape[0]

    # Display batch information
    print(f"Batch Information - Batch Size: 1, Sentences: {n_sentences}, Tokens per Sentence: {n_tokens}, Emotions: {n_emotions}, Triggers: {n_triggers}")
    break

Batch Information - Batch Size: 1, Sentences: 10, Tokens per Sentence: 25, Emotions: 10, Triggers: 10


# Baselines

## Random Classifier

In [9]:
class RandomClassifier:
    def __init__(self, num_emotions: int, seed: int) -> None:
        self.__num_emotions = num_emotions
        np.random.seed(seed)

    def __call__(self, dialogue: List[str]) -> Tuple[List[int], List[int]]:
        emotions = np.random.rand(len(dialogue), self.__num_emotions)
        emotions = emotions / emotions.sum(axis=1, keepdims=True)
        triggers = np.random.randint(2, size=len(dialogue))
        return np.argmax(emotions, axis=1), triggers

random_classifier = RandomClassifier(num_emotions=7, seed=42)
dialogue = ["I feel happy.", "This is a sad moment Hugo, please.", "I am excited!", "Best, Laura!"]
emotions, trigger_probs = random_classifier(dialogue)
print("Emotions:", emotions)
print("Triggers:", trigger_probs)

Emotions: [1 4 6 4]
Triggers: [1 0 0 0]


## Majority Classifier

In [10]:
class MajorityClassifier:
    def __init__(self, data: pd.DataFrame) -> None:
        self.__majority_emotion = self.__calculate_majority_label(data, key="emotions")
        self.__majority_trigger = self.__calculate_majority_label(data, key="triggers")

    def __calculate_majority_label(self, data: pd.DataFrame, key: str) -> int:
        all_lists = [d[1][key] for d in train_data.iterrows()]
        final_list = list(chain(*all_lists))
        most_frequent_item = Counter(final_list).most_common(1)[0][0]
        return int(most_frequent_item)

    def __call__(self, dialogue: List[str]) -> Tuple[List[int], List[int]]:
        emotions = np.tile(self.__majority_emotion, len(dialogue))
        triggers = np.tile(self.__majority_trigger, len(dialogue))
        return emotions, triggers

majority_classifier = MajorityClassifier(train_data)
dialogue = ["I feel happy.", "This is a sad moment Hugo, please.", "I am excited!", "Best, Laura!"]
emotions, trigger_probs = majority_classifier(dialogue)
print("Emotions:", emotions)
print("Triggers:", trigger_probs)

Emotions: [0 0 0 0]
Triggers: [0 0 0 0]


# Our solution: Model definition

This PyTorch model is designed for emotion and trigger prediction in text data. Here's a breakdown of its components:

1. **Initialization:**
   - `EmotionTriggerModel` is a class that inherits from `nn.Module`.
   - It takes several parameters during initialization:
      - `model_name`: A string specifying the name of a pre-trained transformer model (assumed to be a RoBERTa-based model).
      - `num_emotions`: An integer indicating the number of emotion classes for prediction.
      - `lstm_hidden_size`: An integer specifying the hidden size of the LSTM layer for trigger prediction.
      - `ffnn_hidden_size`: An integer specifying the hidden size of the feed-forward neural networks.
      - `random_state`: An integer representing the random seed for reproducibility.

2. **RoBERTa Model and Tokenizer:**
   - It uses a RoBERTa-based model for sequence classification (`AutoModelForSequenceClassification`) with a specified number of output labels (`num_emotions`).
   - The RoBERTa model is expected to return hidden states (`output_hidden_states=True`).
   - The RoBERTa tokenizer (`AutoTokenizer`) is initialized from the specified `model_name`.

3. **Emotion Feed-forward Neural Network:**
   - A feed-forward neural network for emotion prediction is defined using `nn.Sequential`.
   - It consists of a linear layer, ReLU activation, another linear layer, and a softmax activation function for multi-class emotion prediction.

4. **LSTM for Trigger Prediction:**
   - An LSTM layer is defined for trigger prediction.
   - The input size to the LSTM includes the RoBERTa hidden size and an additional dimension for the predicted emotion.
   - It is a bidirectional LSTM (`bidirectional=True`).

5. **Feed-forward Neural Network for Trigger Prediction:**
   - Another feed-forward neural network is defined for trigger prediction after the LSTM layer.
   - It consists of a linear layer, ReLU activation, another linear layer, and a sigmoid activation function for binary trigger prediction.

6. **Forward Method:**
   - The `forward` method takes input IDs, attention mask, and token type IDs as input parameters.
   - It passes the input through the RoBERTa model to obtain hidden states.
   - The CLS token is extracted from the last hidden states for emotion prediction.
   - Emotion prediction is performed using the feed-forward neural network.
   - The predicted emotion is concatenated with the CLS token and passed through the LSTM layer for trigger prediction.
   - The final trigger prediction is obtained using the feed-forward neural network for triggers.
   - The method returns tensors representing predicted emotions and trigger probabilities.

In summary, this model is a combination of a RoBERTa-based emotion classifier and an LSTM-based trigger predictor for text data. It is designed to jointly predict emotions and triggers in a given sequence of text.

In [20]:
class EmotionTriggerModel(nn.Module, PyTorchModelHubMixin):
    def __init__(self, encoder_model_name: str, num_emotions: int, lstm_hidden_size: int, ffnn_hidden_size: int, random_state: int, freeze_encoder: bool = False) -> None:
        super().__init__()
        torch.manual_seed(random_state)
        torch.cuda.manual_seed_all(random_state)

        # BERT model for emotion prediction
        self.roberta_model = AutoModelForSequenceClassification.from_pretrained(encoder_model_name, num_labels=num_emotions, output_hidden_states=True)
        if freeze_encoder:
            for param in self.roberta_model.parameters():
                param.requires_grad = False

        # Feed-forward neural network for emotion prediction
        self.emotion_ffnn = nn.Sequential(
            nn.Linear(self.roberta_model.config.hidden_size, ffnn_hidden_size),
            nn.ReLU(),
            nn.Linear(ffnn_hidden_size, num_emotions),
        )

        # LSTM for trigger prediction
        self.lstm = nn.LSTM(input_size=self.roberta_model.config.hidden_size + 1, # add the predicted emotion
                            hidden_size=lstm_hidden_size,
                            batch_first=True,
                            bidirectional=True)

        # Feed-forward neural network for trigger prediction
        self.trigger_ffnn = nn.Sequential(
            nn.Linear(lstm_hidden_size * 2, ffnn_hidden_size),
            nn.ReLU(),
            nn.Linear(ffnn_hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
      # Get BERT embeddings for each sentence
      roberta_outputs = self.roberta_model(**{"input_ids": input_ids, "attention_mask": attention_mask}).hidden_states[-1]

      # Extract CLS token for emotion prediction
      cls_tokens = roberta_outputs[:, 0, :]

      # Emotion prediction
      emotions_logits = self.emotion_ffnn(cls_tokens)

      # Concatenate predicted emotions (softmax + argmax) with CLS tokens
      emotions = torch.argmax(nn.functional.softmax(emotions_logits, dim=1), dim=1)
      concat_inputs = torch.cat((cls_tokens, emotions.unsqueeze(1)), dim=1)

      # LSTM for trigger prediction
      lstm_outputs, _ = self.lstm(concat_inputs)

      # Feed-forward neural network for trigger prediction
      trigger_probs = self.trigger_ffnn(lstm_outputs)

      return emotions_logits, trigger_probs.squeeze()

## Example usage

In [22]:
encoder_model_name = "roberta-base"
model = EmotionTriggerModel(encoder_model_name, num_emotions=7, lstm_hidden_size=64, ffnn_hidden_size=32, random_state=random_state)
roberta_tokenizer = AutoTokenizer.from_pretrained(encoder_model_name)
dialogue = ["I feel happy.", "This is a sad moment Hugo, please.", "I am excited!", "Best, Laura!"]
emotions, trigger_probs = model(**roberta_tokenizer(dialogue, return_tensors='pt', padding=True, truncation=True))
print("Emotion logits:", emotions)
print("Trigger Probabilities:", trigger_probs)

Emotion logits: tensor([[-0.1195, -0.0462, -0.2287,  0.0244, -0.0431,  0.0193, -0.1269],
        [-0.1280, -0.0452, -0.2351,  0.0316, -0.0525,  0.0148, -0.1244],
        [-0.1435, -0.0536, -0.2251,  0.0174, -0.0490,  0.0047, -0.1292],
        [-0.1452, -0.0496, -0.2304,  0.0162, -0.0595,  0.0139, -0.1264]],
       grad_fn=<AddmmBackward0>)
Trigger Probabilities: tensor([0.4951, 0.4993, 0.5041, 0.5078], grad_fn=<SqueezeBackward0>)


# Our solution: Training

Execute training with a batch size of 1, meaning one dialogue is processed at a time. The model will be fed tokenized dialogues, including attention masks and token type IDs, resulting in a PyTorch tensor with a shape of `[batch_size, n_sentences, n_tokens_per_sentence]`. This tensor is then passed through the model (after unsqueezing it to remove the first dimension). The model returns two results: emotions and triggers. The loss is computed, and the parameters are updated through backward propagation.

In [42]:
@torch.inference_mode()
def evaluate_model(model, eval_dataloader, epoch_number, criterion_emotion, criterion_trigger, device) -> float:
    model.eval()
    total_loss: float = 0.0
    with torch.no_grad():
        for idx, batch in enumerate(tqdm(eval_dataloader, desc=f"Evaluating after epoch {epoch_number}")):
          input_ids = batch['input_ids'].to(device)
          attention_mask = batch['attention_mask'].to(device)
          emotion_labels = batch['emotions'].to(device)
          trigger_labels = batch['triggers'].to(device)

          emotion_logits, trigger_probs = model(input_ids=input_ids, attention_mask=attention_mask)

          emotion_loss = criterion_emotion(emotion_logits, emotion_labels)
          trigger_loss = criterion_trigger(trigger_probs, trigger_labels)

          loss = 1 * emotion_loss + .1 * trigger_loss # penalize more the emotion, since it holds higher loss values
          total_loss += loss.item()

    return total_loss / len(eval_dataloader)

def train_model(model, num_epochs, train_dataloader, eval_dataloader, optimizer, scheduler, criterion_emotion, criterion_trigger, device, model_url: str) -> Dict[str, List[float]]:
    model.to(device)
    train_losses: List[float] = list()
    eval_losses: List[float] = list()
    best_eval_loss = float('inf')

    for epoch in range(num_epochs):
      model.train()
      total_loss: float = 0.0
      for idx, batch in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch+1}")):
          input_ids = batch['input_ids'].to(device)
          attention_mask = batch['attention_mask'].to(device)
          emotion_labels = batch['emotions'].to(device)
          trigger_labels = batch['triggers'].to(device)

          optimizer.zero_grad()

          emotion_logits, trigger_probs = model(input_ids=input_ids, attention_mask=attention_mask)

          emotion_loss = criterion_emotion(emotion_logits, emotion_labels)
          trigger_loss = criterion_trigger(trigger_probs, trigger_labels)

          loss = 1 * emotion_loss + .1 * trigger_loss # penalize more the emotion, since it holds higher loss values
          total_loss += loss.item()

          loss.backward()
          optimizer.step()
          scheduler.step()

      avg_train_loss: float = total_loss / len(train_dataloader)
      train_losses.append(avg_train_loss)
      print(f"Epoch {epoch+1}/{num_epochs}, Avg Train Loss: {avg_train_loss:.4f}")

      avg_eval_loss: float = evaluate_model(model, eval_dataloader, epoch+1, criterion_emotion, criterion_trigger, device)
      eval_losses.append(avg_eval_loss)
      print(f"Epoch {epoch+1}/{num_epochs}, Avg Validation Loss: {avg_eval_loss:.4f}")

      # Save the best model
      if avg_eval_loss < best_eval_loss:
        save_to = model_url + f"_epoch-{epoch+1}"
        print(f"Saving model to: {save_to}")
        best_eval_loss = avg_eval_loss
        model.save_pretrained(save_to)
        model.push_to_hub(save_to)

    return {"train_loss_history": train_losses, "eval_loss_history": eval_losses}

In [ ]:
seeds: List[int] = [42, 64, 512, 1024, 4096]
n_epochs: int = 8
learning_rate: float = 2e-5
criterion_emotion = nn.CrossEntropyLoss()
criterion_trigger = nn.BCELoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder_model_name = "roberta-base"
roberta_model_hub_path: str = f"balzanilo/dialogs_{encoder_model_name}"
history_list: List = list()

for seed in seeds:
  model = EmotionTriggerModel(encoder_model_name, num_emotions=7, lstm_hidden_size=64, ffnn_hidden_size=32, random_state=seed)
  optimizer = AdamW(model.parameters(), lr=learning_rate)
  scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=len(train_data)*n_epochs)
  history = train_model(model, n_epochs, train_dataloader, eval_dataloader, optimizer, scheduler, criterion_emotion, criterion_trigger, device, model_url = roberta_model_hub_path + f"_seed-{seed}")
  history_list.append(history)

Epoch 0: 100%|██████████| 3200/3200 [04:54<00:00, 10.88it/s]


Epoch 1/8, Avg Train Loss: 1.1478


Evaluating after epoch 0: 100%|██████████| 400/400 [00:07<00:00, 51.27it/s]


Epoch 1/8, Avg Validation Loss: 0.7593
Saving model to: balzanilo/dialogs_roberta-base_seed-42_epoch-0


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

...

Epoch 2/8, Avg Train Loss: 0.6307


Evaluating after epoch 1: 100%|██████████| 400/400 [00:07<00:00, 54.75it/s]


Epoch 2/8, Avg Validation Loss: 0.4686
Saving model to: balzanilo/dialogs_roberta-base_seed-512_epoch-1


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Epoch 2:  37%|███▋      | 1178/3200 [01:44<03:02, 11.07it/s]

# Our solution: Testing

In [109]:
@torch.inference_mode()
def inference(model, input, device: str):
  model.eval()
  model.to(device)
  with torch.no_grad():
    emotions, trigger_probs = model(input_ids=input["input_ids"].to(device), attention_mask=input["attention_mask"].to(device))
  print("Emotions:", emotions)
  print("Ground truth emotions:", input["emotions"])
  print("Trigger Probabilities:", trigger_probs)

model = EmotionTriggerModel.from_pretrained(f"{roberta_model_hub_path}_seed-64-epoch-8")
inference(model, test_dataset[0], device)

Emotions: tensor([[ 0.0793, -0.5747, -0.0876,  0.2002,  0.1658, -0.3628,  0.2384],
        [ 0.1297, -0.6071, -0.0627,  0.2262,  0.1228, -0.3564,  0.2506],
        [ 0.0640, -0.5404, -0.0946,  0.1976,  0.2011, -0.3756,  0.1970],
        [ 0.1216, -0.6025, -0.0452,  0.2152,  0.1305, -0.3834,  0.2312],
        [ 0.0915, -0.5729, -0.1047,  0.1888,  0.1908, -0.4173,  0.2158],
        [ 0.1952, -0.6640,  0.0674,  0.1904,  0.0299, -0.3235,  0.2885],
        [ 0.0662, -0.5835, -0.0695,  0.2005,  0.2031, -0.3910,  0.2131],
        [ 0.1362, -0.6294, -0.0365,  0.2193,  0.1383, -0.3475,  0.2704]],
       device='cuda:0')
Ground truth emotions: tensor([0, 0, 0, 0, 0, 0, 0, 3])
Trigger Probabilities: tensor([0.4747, 0.4700, 0.4626, 0.4642, 0.4605, 0.4676, 0.4669, 0.4672],
       device='cuda:0')


# Our solution: Evaluation

In [95]:
def compute_f1_scores(y_pred: List, y_test: List):
    if len(y_pred) != len(y_test):
        raise ValueError("Predictions and labels must be of the same length")

    # Sequence F1
    sequence_f1_scores = []
    for pred, true in zip(y_pred, y_test):
        if len(pred) != len(true):
            raise ValueError("Prediction and label dialogue lengths must match")
        sequence_f1_scores.append(f1_score(true, pred, average='macro'))

    avg_sequence_f1 = np.mean(sequence_f1_scores)

    # Unrolled Sequence F1
    all_pred = [item for sublist in y_pred for item in sublist]
    all_test = [item for sublist in y_test for item in sublist]
    unrolled_sequence_f1 = f1_score(all_test, all_pred, average='macro')

    return avg_sequence_f1, unrolled_sequence_f1

In [97]:
emotions_predictions_list = list()
emotions_labels_list = list()
triggers_predictions_list = list()
triggers_labels_list = list()

model.eval()
with torch.no_grad():
    for batch in tqdm(test_dataloader):
      input_ids = batch['input_ids'].to(device)
      attention_mask = batch['attention_mask'].to(device)
      token_type_ids = batch["token_type_ids"].to(device)
      emotions_labels_list.append(batch['emotions'])
      triggers_labels_list.append(batch['triggers'])

      emotions, trigger_probs = model(input_ids=input_ids, attention_mask=attention_mask)
      emotions_probabilities = torch.softmax(emotions, dim=1)
      emotion_predictions = torch.argmax(emotions_probabilities, dim=1)
      emotions_predictions_list.append(emotion_predictions.cpu().numpy())
      triggers_prediction = (trigger_probs > .5).int()
      triggers_predictions_list.append(triggers_prediction.cpu().numpy())

100%|██████████| 400/400 [00:25<00:00, 15.62it/s]


In [ ]:
emotions_sequence_f1, emotions_unrolled_sequence_f1 = compute_f1_scores(emotions_predictions_list, emotions_labels_list)
print(f"\nSequence F1: {emotions_sequence_f1:.2%}, Unrolled Sequence F1: {emotions_unrolled_sequence_f1:.2%}")


Sequence F1: 90.56%, Unrolled Sequence F1: 93.05%


In [ ]:
triggers_sequence_f1, triggers_unrolled_sequence_f1 = compute_f1_scores(triggers_predictions_list, triggers_labels_list)
print(f"\nSequence F1: {triggers_sequence_f1:.2%}, Unrolled Sequence F1: {triggers_unrolled_sequence_f1:.2%}")


Sequence F1: 60.18%, Unrolled Sequence F1: 64.61%


# Conclusions: Our solution vs Baselines